In [1]:
import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="fintech_reviews",
    user="postgres",
    password="your_password_here"  # replace with your password
)
cursor = conn.cursor()
print("Connected to database!")

Connected to database!


In [2]:
df = pd.read_csv('../data/raw/reviews_with_sentiment.csv')
print(f"Loaded {len(df)} reviews")
print(df.columns.tolist())

Loaded 1478 reviews
['review', 'rating', 'date', 'bank', 'source', 'sentiment_label', 'sentiment_score']


In [6]:
cursor.execute("DELETE FROM reviews")
conn.commit()
print("leared reviews table!")

leared reviews table!


In [7]:
cursor.execute("SELECT id, name FROM banks")
banks = cursor.fetchall()
bank_map = {name: id for id, name in banks}

def clean_text(text):
    """Remove emojis and special characters"""
    return str(text).encode('ascii', 'ignore').decode('ascii')

count = 0
for _, row in df.iterrows():
    try:
        bank_id = bank_map.get(row['bank'])
        theme = str(row['theme']) if 'theme' in df.columns else 'General'
        cursor.execute("""
            INSERT INTO reviews 
            (bank_id, review, rating, date, sentiment_label, sentiment_score, theme, source)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            bank_id,
            clean_text(row['review']),
            int(row['rating']),
            str(row['date']),
            str(row['sentiment_label']),
            float(row['sentiment_score']),
            theme,
            str(row['source'])
        ))
        count += 1
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

conn.commit()
print(f"Inserted {count} reviews!")

Inserted 1478 reviews!


In [8]:
cursor.execute("""
    SELECT b.name, COUNT(r.id) as review_count
    FROM reviews r
    JOIN banks b ON r.bank_id = b.id
    GROUP BY b.name
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]}: {row[1]} reviews")

cursor.close()
conn.close()
print("Connection closed!")

Dashen Bank: 498 reviews
Bank of Abyssinia: 498 reviews
Commercial Bank of Ethiopia: 482 reviews
Connection closed!
